# ETL with the DataFrame API

**ETL** (extract, transform, load) is the bread and butter of data
engineering: pull data out of a source, clean and reshape it, and store the
result where analysts, dashboards and models can use it. This page builds a
small but complete ETL pipeline over the NYC taxi sample data. It uses the
DataFrame operations from
{doc}`Introduction to DataFrames <../02_spark_dataframes_and_sql/01_introduction_to_dataframes>`,
the readers and writers from
{doc}`Reading and Writing Data <../02_spark_dataframes_and_sql/02_reading_and_writing_data>`,
and the performance ideas from
{doc}`Distributed Systems Programming Fundamentals <../03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.

## The Medallion Architecture

Pipelines on Databricks commonly organise tables into three layers of
increasing quality, known as the **medallion architecture**
([What is the medallion lakehouse architecture?](https://docs.databricks.com/aws/en/lakehouse/medallion)):

```mermaid
:caption: Bronze, silver and gold layers of the pipeline built on this page.
:align: center

graph LR
    Src[("samples.nyctaxi.trips")] --> B[Bronze: etl04_trips_bronze]
    B --> S[Silver: etl04_trips_silver]
    S --> G[Gold: etl04_daily_zip_revenue]
```

- **Bronze**: raw, unvalidated data, with only minimal validation and
  metadata columns such as the source of the data. Retaining all of it
  enables reprocessing and auditing
  ([medallion: bronze](https://docs.databricks.com/aws/en/lakehouse/medallion#ingest-raw-data-to-the-bronze-layer)).
- **Silver**: validated, cleaned and enriched data. This is where you perform
  cleansing, de-duplication and normalisation, and it keeps at least one
  validated, non-aggregated representation of each record
  ([medallion: silver](https://docs.databricks.com/aws/en/lakehouse/medallion#build-silver-tables-from-the-bronze-layer)).
- **Gold**: aggregated data tailored for analytics and reporting, mapped to
  business functions and needs
  ([medallion: gold](https://docs.databricks.com/aws/en/lakehouse/medallion#power-analytics-with-the-gold-layer)).

The examples write to `workspace.default`, with table names prefixed
`etl04_`; substitute your own catalog and schema if needed. The first cell
drops the tables so the page can be re-run:

In [1]:
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

TARGET = "workspace.default"
TABLES = ["etl04_trips_bronze", "etl04_trips_silver", "etl04_daily_zip_revenue"]

for table in TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {TARGET}.{table}")

## Extract: Building the Bronze Layer

The extract step reads the source as faithfully as possible and records
where and when each row came from:

In [2]:
def extract_trips(source_table: str) -> DataFrame:
    return (
        spark.read.table(source_table)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source", F.lit(source_table))
    )


bronze = extract_trips("samples.nyctaxi.trips")
bronze.write.mode("overwrite").saveAsTable(f"{TARGET}.etl04_trips_bronze")
spark.read.table(f"{TARGET}.etl04_trips_bronze").count()

21932

When the source is a set of files in a volume instead of a table, the same
function would use `spark.read` with an explicit schema and record
`_metadata.file_path` as the source, as shown in
{doc}`Reading and Writing Data <../02_spark_dataframes_and_sql/02_reading_and_writing_data>`.

:::{admonition} Note: Incremental Ingestion
:class: note

This page reloads the whole source on every run, which is fine for a sample
table. For sources that keep growing, you would ingest only *new* data. Auto
Loader incrementally processes new data files as they arrive in cloud
storage, and Databricks recommends it in Lakeflow pipelines for incremental
ingestion
([What is Auto Loader?](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)).
Lakeflow pipelines extend Apache Spark Declarative Pipelines, a declarative
framework for batch and streaming pipelines
([Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/)).
Both build on the same DataFrame transformations shown here, and
{doc}`Stream Processing and Analysis <../../03_stream_processing_and_analysis/index>`
covers the streaming side.
:::

## Transform: Building the Silver Layer

Transformations are where most of the logic lives. Write each one as a
function from DataFrame to DataFrame, so that each step can be named, tested
on its own and reused.

### Cleaning and Validating

Raw taxi data can contain trips with zero distance, non-positive fares and
drop-off times before pick-up times. Keep only rows that make physical sense:

In [3]:
def filter_valid_trips(df: DataFrame) -> DataFrame:
    return df.filter(
        (F.col("trip_distance") > 0)
        & (F.col("fare_amount") > 0)
        & (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
        & F.col("pickup_zip").isNotNull()
        & F.col("dropoff_zip").isNotNull()
    )


raw = spark.read.table(f"{TARGET}.etl04_trips_bronze")
print("bronze rows:", raw.count(), " valid rows:", filter_valid_trips(raw).count())

bronze rows: 21932  valid rows: 21847


Instead of silently dropping bad rows, a production pipeline usually counts
them or writes them to a separate quarantine table, so data problems are
visible.

### Deriving Columns

Add the columns downstream consumers will need, computed once, here:

In [4]:
def add_trip_features(df: DataFrame) -> DataFrame:
    duration_minutes = (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    ) / 60

    return df.select(
        "*",
        F.to_date("tpep_pickup_datetime").alias("pickup_date"),
        F.round(duration_minutes, 2).alias("duration_minutes"),
        F.round(
            F.col("trip_distance") / (duration_minutes / 60), 2
        ).alias("avg_speed_mph"),
    )

Using a single `select` that adds all three columns keeps the plan small,
as recommended in the
{doc}`Common Mistakes <../03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`
of the distributed systems page.

### De-duplicating with a Window Function

Sources often deliver the same record more than once. `dropDuplicates`
removes duplicate rows, optionally considering only certain columns, but its
documentation doesn't say *which* row of a group it keeps
([Apache Spark API reference: DataFrame.dropDuplicates](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)).
When you need to control which row survives, for example the most recently
ingested one, use a **window function**:

In [5]:
def deduplicate_trips(df: DataFrame) -> DataFrame:
    trip_key = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "pickup_zip", "dropoff_zip"]
    latest_first = Window.partitionBy(*trip_key).orderBy(F.desc("_ingested_at"))

    return (
        df
        .withColumn("_row_number", F.row_number().over(latest_first))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

A window function operates on a group of rows, the *window*, and calculates
a value for each row based on that group
([Apache Spark: Window Functions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)),
without collapsing the group the way `groupBy` does. Like `groupBy`,
`partitionBy` needs all rows of a key together, so it shuffles on the key
columns.

:::::{dropdown} Example: How `row_number` Picks the Survivor
:open:
:color: primary
:icon: pencil
:class-body: example-body

Suppose the same trip arrived twice, in two loads an hour apart.

**Input rows sharing one trip key.**

```text
pickup_zip  fare_amount  _ingested_at
10001       12.5         2026-10-12 08:00
10001       12.5         2026-10-12 09:00
```

**Number the rows within the window, newest first.**

```text
pickup_zip  fare_amount  _ingested_at       _row_number
10001       12.5         2026-10-12 09:00   1
10001       12.5         2026-10-12 08:00   2
```

Keeping `_row_number == 1` retains exactly one row per trip, the 09:00 copy.
The cell below runs `deduplicate_trips` on these two rows.
:::::

In [6]:
two_loads = spark.createDataFrame(
    [
        ("2016-02-01 10:00", "2016-02-01 10:15", 10001, 10003, 12.5, "2026-10-12 08:00"),
        ("2016-02-01 10:00", "2016-02-01 10:15", 10001, 10003, 12.5, "2026-10-12 09:00"),
    ],
    schema="""
        tpep_pickup_datetime STRING, tpep_dropoff_datetime STRING,
        pickup_zip INT, dropoff_zip INT, fare_amount DOUBLE, _ingested_at STRING
    """,
).withColumns(
    {c: F.to_timestamp(c) for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime", "_ingested_at"]}
)
deduplicate_trips(two_loads).select("pickup_zip", "fare_amount", "_ingested_at").show()

+----------+-----------+-------------------+
|pickup_zip|fare_amount|       _ingested_at|
+----------+-----------+-------------------+
|     10001|       12.5|2026-10-12 09:00:00|
+----------+-----------+-------------------+



### Composing the Steps

`DataFrame.transform` gives a concise syntax for chaining functions that take
and return a DataFrame
([Apache Spark API reference: DataFrame.transform](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.transform.html)),
so the whole silver step reads as a list of named stages:

In [7]:
def build_silver(bronze: DataFrame) -> DataFrame:
    return (
        bronze
        .transform(filter_valid_trips)
        .transform(add_trip_features)
        .transform(deduplicate_trips)
    )


silver = build_silver(spark.read.table(f"{TARGET}.etl04_trips_bronze"))
silver.write.mode("overwrite").saveAsTable(f"{TARGET}.etl04_trips_silver")
spark.read.table(f"{TARGET}.etl04_trips_silver").count()

21847

Silver has 21,847 rows: the 85 invalid trips were filtered out, and the sample
data contained no duplicate trips for `deduplicate_trips` to remove.

Nothing runs until `saveAsTable`. At that point Spark optimises the filter,
the derived columns and the de-duplication as one plan, as described in
{doc}`Introduction to DataFrames <../02_spark_dataframes_and_sql/01_introduction_to_dataframes>`.

## Transform: Building the Gold Layer

Gold tables answer a specific question. Here: how much fare revenue did each
pick-up zip code generate per day, and what did a typical trip look like?

In [8]:
def build_daily_zip_revenue(silver: DataFrame) -> DataFrame:
    return (
        silver
        .groupBy("pickup_date", "pickup_zip")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("fare_amount"), 2).alias("total_fare"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
            F.round(F.percentile_approx("duration_minutes", 0.5), 2).alias("median_duration_minutes"),
        )
    )


gold = build_daily_zip_revenue(spark.read.table(f"{TARGET}.etl04_trips_silver"))
gold.write.mode("overwrite").saveAsTable(f"{TARGET}.etl04_daily_zip_revenue")
spark.read.table(f"{TARGET}.etl04_daily_zip_revenue").orderBy(F.desc("total_fare")).show(5)

+-----------+----------+----------+----------+------------+-----------------------+
|pickup_date|pickup_zip|trip_count|total_fare|avg_distance|median_duration_minutes|
+-----------+----------+----------+----------+------------+-----------------------+
| 2016-02-12|     11422|        15|     637.5|       15.25|                   41.3|
| 2016-01-14|     11422|        14|     597.5|       14.54|                  29.52|
| 2016-01-14|     11371|        17|     545.0|       10.48|                  29.83|
| 2016-01-18|     11422|        12|     522.0|       15.72|                  30.85|
| 2016-01-13|     11422|        11|     505.5|       15.13|                  37.88|
+-----------+----------+----------+----------+------------+-----------------------+
only showing top 5 rows


`percentile_approx` returns an *approximate* percentile within the group
([percentile_approx aggregate function](https://docs.databricks.com/aws/en/sql/language-manual/functions/percentile_approx)),
here the median trip duration. An exact median needs all values of a group in
one place; the approximate version trades a little accuracy for a much
cheaper computation.

## Load: Writing Results Safely

Every run of a pipeline should leave the target in a correct state, even if
the run is repeated. A pipeline with this property is **idempotent**, and
it lets you safely retry after a failure. The choice of write strategy
decides whether a pipeline is idempotent:

| Strategy | When to use it | Idempotent? |
| --- | --- | --- |
| `mode("overwrite")` | Small tables you can rebuild completely | Yes |
| `mode("overwrite")` with `replaceWhere` | Rebuilding one slice, such as one day | Yes |
| `mode("append")` | Adding new, never-seen-before rows | No: a retry duplicates rows |
| `MERGE INTO` | Inserting new rows and updating changed ones | Yes, when keyed correctly |

`replaceWhere` atomically overwrites the rows that match a predicate, and
the table doesn't need to be partitioned on the predicate's columns
([Selectively overwrite data with Delta Lake](https://docs.databricks.com/aws/en/delta/selective-overwrite#replace-where)).
Overwriting a single day of the gold table:

In [9]:
day = "2016-02-13"
print("rows before:", spark.read.table(f"{TARGET}.etl04_daily_zip_revenue").count())

(
    gold
    .filter(F.col("pickup_date") == day)
    .write
    .mode("overwrite")
    .option("replaceWhere", f"pickup_date = '{day}'")
    .saveAsTable(f"{TARGET}.etl04_daily_zip_revenue")
)
print("rows after: ", spark.read.table(f"{TARGET}.etl04_daily_zip_revenue").count())

rows before: 3271


rows after:  3271


The row count is unchanged: the day's rows were replaced by identical ones,
so running the cell again leaves the same table.

An **upsert** with `MERGE INTO` updates rows that already exist and inserts
the rest. `UPDATE SET *` and `INSERT *` assume the source has the same
columns as the target
([Upsert into a Delta Lake table using merge](https://docs.databricks.com/aws/en/delta/merge#merge-operation-semantics)).
The new data is exposed to SQL as a temporary view:

In [10]:
gold.createOrReplaceTempView("etl04_daily_zip_revenue_updates")

spark.sql(f"""
    MERGE INTO {TARGET}.etl04_daily_zip_revenue AS target
    USING etl04_daily_zip_revenue_updates AS source
    ON  target.pickup_date = source.pickup_date
    AND target.pickup_zip  = source.pickup_zip
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             3271|            3271|               0|                0|
+-----------------+----------------+----------------+-----------------+



The result reports how many rows were updated and inserted. Since `gold`
holds exactly the rows already in the table, every row matched and none was
inserted.

`MERGE` runs as a single Delta transaction, and Delta transactions either
succeed or fail completely
([ACID guarantees on Databricks](https://docs.databricks.com/aws/en/lakehouse/acid#atomicity)),
so readers never see a half-merged table.

## Checking the Result

A few cheap checks after each load catch most problems before anyone else
notices them:

In [11]:
silver = spark.read.table(f"{TARGET}.etl04_trips_silver")

checks = silver.agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("fare_amount") <= 0, 1).otherwise(0)).alias("non_positive_fares"),
    F.sum(F.when(F.col("duration_minutes") <= 0, 1).otherwise(0)).alias("non_positive_durations"),
).first()

assert checks["rows"] > 0, "silver table is empty"
assert checks["non_positive_fares"] == 0
assert checks["non_positive_durations"] == 0
checks

Row(rows=21847, non_positive_fares=0, non_positive_durations=0)

All three numbers come back in a single `Row` from a single query, because
they are computed in one `agg`. Running three separate `count()` actions
would scan the table three times.

Delta tables also keep a history of every write, with the operation, user
and timestamp of each one
([Work with table history](https://docs.databricks.com/aws/en/delta/history#retrieve-table-history)):

In [12]:
(
    spark.sql(f"DESCRIBE HISTORY {TARGET}.etl04_daily_zip_revenue")
    .select("version", "operation", "operationMetrics.numOutputRows")
    .show(truncate=False)
)

+-------+---------------------------------+-------------+
|version|operation                        |numOutputRows|
+-------+---------------------------------+-------------+
|2      |MERGE                            |3271         |
|1      |WRITE                            |3271         |
|0      |CREATE OR REPLACE TABLE AS SELECT|3271         |
+-------+---------------------------------+-------------+



## Testing Transformations

Because each transformation is a plain function from DataFrame to DataFrame,
it can be unit tested with a tiny hand-built input. This repository has a
`pytest` setup under the `test` dependency group. Assuming the functions
above live in a module such as `project/etl/trips.py`, a test looks like this:

```python
from datetime import datetime

from project.utils.sparksession import get_spark_session

# wherever you keep the pipeline's transformations
from project.etl.trips import filter_valid_trips


def test_filter_valid_trips_drops_negative_fares():
    spark = get_spark_session(profile="free-edition")
    df = spark.createDataFrame(
        [
            (datetime(2016, 2, 1, 10, 0), datetime(2016, 2, 1, 10, 15), 2.0, 9.5, 10001, 10003),
            (datetime(2016, 2, 1, 11, 0), datetime(2016, 2, 1, 11, 10), 1.5, -3.0, 10001, 10003),
        ],
        schema="""
            tpep_pickup_datetime TIMESTAMP, tpep_dropoff_datetime TIMESTAMP,
            trip_distance DOUBLE, fare_amount DOUBLE,
            pickup_zip INT, dropoff_zip INT
        """,
    )

    result = filter_valid_trips(df)

    assert result.count() == 1
```

The first row is a valid 15-minute trip and passes the filter; the second has
a negative fare and is dropped, so exactly one row remains. The same check
runs here against the function defined on this page:

In [13]:
from datetime import datetime

test_input = spark.createDataFrame(
    [
        (datetime(2016, 2, 1, 10, 0), datetime(2016, 2, 1, 10, 15), 2.0, 9.5, 10001, 10003),
        (datetime(2016, 2, 1, 11, 0), datetime(2016, 2, 1, 11, 10), 1.5, -3.0, 10001, 10003),
    ],
    schema="""
        tpep_pickup_datetime TIMESTAMP, tpep_dropoff_datetime TIMESTAMP,
        trip_distance DOUBLE, fare_amount DOUBLE,
        pickup_zip INT, dropoff_zip INT
    """,
)
filter_valid_trips(test_input).count()

1

## When to Use This Pattern, and the Alternatives

| Approach | Use it when | Trade-off |
| --- | --- | --- |
| Batch DataFrame ETL, as on this page | The source is small enough to reprocess, or arrives in batches | Each run re-reads everything it is given |
| Structured Streaming with Auto Loader | Files keep arriving and you want to process only the new ones | You manage checkpoints and the stream's lifecycle |
| Lakeflow pipelines | You want the framework to manage dependencies between tables, incremental processing and infrastructure | Pipelines are defined declaratively, in a pipeline rather than a notebook |

Databricks recommends streaming reads from bronze for append-only sources,
using batch reads only for small datasets
([medallion: silver](https://docs.databricks.com/aws/en/lakehouse/medallion#build-silver-tables-from-the-bronze-layer)),
and Auto Loader in Lakeflow pipelines for incremental ingestion
([What is Auto Loader?](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/)).

## Common Mistakes

- **Writing to silver straight from the source.** Databricks does not
  recommend writing to silver tables directly from ingestion
  ([medallion: silver](https://docs.databricks.com/aws/en/lakehouse/medallion#build-silver-tables-from-the-bronze-layer)).
  Land the raw data in bronze first, so it can be reprocessed when the
  cleaning logic changes.
- **Relying on `dropDuplicates` to keep a particular row.** Its documentation
  doesn't specify which row survives
  ([Apache Spark API reference: DataFrame.dropDuplicates](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)).
  Use `row_number` over a window with an explicit order.
- **Merging a source with duplicate keys.** Only a single source row can
  match a given target row; otherwise the update is ambiguous and the merge
  fails
  ([Upsert into a Delta Lake table using merge](https://docs.databricks.com/aws/en/delta/merge#merge-operation-semantics)).
  Here the updates contain the same day and zip code twice:

In [14]:
gold.unionByName(gold).createOrReplaceTempView("etl04_duplicated_updates")

spark.sql(f"""
    MERGE INTO {TARGET}.etl04_daily_zip_revenue AS target
    USING etl04_duplicated_updates AS source
    ON  target.pickup_date = source.pickup_date
    AND target.pickup_zip  = source.pickup_zip
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()

UnsupportedOperationException: [DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE] Cannot perform Merge as multiple source rows matched and attempted to modify the same
target row in the Delta table in possibly conflicting ways. By SQL semantics of Merge,
when multiple source rows match on the same target row, the result may be ambiguous
as it is unclear which source row should be used to update or delete the matching
target row. You can preprocess the source table to eliminate the possibility of
multiple matches. Please refer to
https://docs.databricks.com/delta/merge.html#merge-error


JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaUnsupportedOperationException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.multipleSourceRowMatchingTargetRowInMergeException(DeltaErrorsEdge.scala:1461)
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.multipleSourceRowMatchingTargetRowInMergeException$(DeltaErrorsEdge.scala:1456)
	at com.databricks.sql.transaction.taho

  The error is `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`.
  De-duplicate the source on the merge key before merging.
- **Writing rows outside the `replaceWhere` predicate.** By default the
  write fails if any of its rows fall outside the predicate
  ([Selectively overwrite data with Delta Lake](https://docs.databricks.com/aws/en/delta/selective-overwrite#replace-where)).
  Filter the DataFrame with the same condition, as the example above does:

In [15]:
(
    gold
    .write
    .mode("overwrite")
    .option("replaceWhere", f"pickup_date = '{day}'")
    .saveAsTable(f"{TARGET}.etl04_daily_zip_revenue")
)

AnalysisException: [DELTA_REPLACE_WHERE_MISMATCH.INVARIANT_VIOLATION] Written data does not conform to partial table overwrite condition or constraint 'pickup_date = '2016-02-13''. [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint EXPRESSION(('pickup_date = 2016-02-13)) (pickup_date = '2016-02-13') violated by row with values:
 - pickup_date : 2016-01-01.


JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsBase.replaceWhereMismatchException(DeltaErrors.scala:1402)
	at com.databricks.sql.transaction.tahoe.DeltaErrorsBase.replaceWhereMismatchException$(DeltaErrors.scala:1399)
	at com.databricks.sql.transaction.tahoe.DeltaErrors$.replaceWhereMismatchException(DeltaErrors.scala:4791)
	at com.databricks.sql.transaction.tahoe.commands.AtomicReplaceInsertRows$AtomicReplaceInsertRowsJob.liftedTree1$1(DeltaInsertAtomicReplaceExecutor.scala:366)
	at com.databricks.sql.transaction.tahoe.commands.AtomicReplaceInser

  The error is `DELTA_REPLACE_WHERE_MISMATCH` and names a row outside the
  predicate.
- **Using `append` in a pipeline that may be retried.** A rerun appends the
  same rows again. Use `overwrite`, `replaceWhere` or `MERGE` so a retry
  leaves the table in the same state.

## Cleaning Up

Drop the tables created on this page when you are finished:

In [16]:
for table in TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {TARGET}.{table}")

This completes the introduction to Apache Spark. The next chapter,
{doc}`Developing Applications with Apache Spark <../../02_developing_applications_with_apache_spark/index>`,
goes deeper into the DataFrame API itself.

## Resources

- [What is the medallion lakehouse architecture? (Databricks)](https://docs.databricks.com/aws/en/lakehouse/medallion)
- [Upsert into a Delta Lake table using merge (Databricks)](https://docs.databricks.com/aws/en/delta/merge)
- [Selectively overwrite data with Delta Lake (Databricks)](https://docs.databricks.com/aws/en/delta/selective-overwrite)
- [Window functions (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)